In [11]:
import os
from sklearn.model_selection import train_test_split

img_path = "/wecare/projects/Slicer_ready_data/Original Patient Data/Images_nii/"
label_path = "/wecare/projects/Slicer_ready_data/Original Patient Data/Corrected_annotations/nifti/"

img_files = os.listdir(img_path)
label_files = os.listdir(label_path)

img_files.sort()
label_files.sort()

print(len(img_files), len(label_files))
for image, label in zip(img_files, label_files):
    #print(os.path.basename(image).split('_')[0],  os.path.basename(image).split('_')[1], os.path.basename(label).split('_')[0], os.path.basename(label).split('_')[1].split('.')[0])
    if os.path.basename(image).split('_')[0] + os.path.basename(image).split('_')[1] != os.path.basename(label).split('_')[0] + os.path.basename(label).split('_')[1].split('.')[0]:
        img_files.remove(image)
        #print(image)

print(len(img_files), len(label_files))

#splitting
img_train, img_val_test, label_train, label_val_test = train_test_split(img_files, label_files, test_size=0.3, random_state=42)
img_val, img_test, label_val, label_test = train_test_split(img_val_test, label_val_test, test_size=0.5, random_state=42)
print(len(img_train), len(img_val), len(img_test))
print(len(label_train), len(label_val), len(label_test))

96 84
84 84
58 13 13
58 13 13


In [ ]:
import os
import numpy as np
import nibabel as nib
from utils import apply_windowing
from random import randint
#import matplotlib.pyplot as plt
from skimage import measure
from skimage.transform import resize
import random

#reshaping the image to 672
#input an image where dimension 1 and 2 are height and width. output the image reshaped to 224x224xN
def reshape(img):

    new_size = 672
    height = img.shape[0]
    width = img.shape[1]
    padding_value = img.min()

    #print(img.shape)
    #img = resize(img, (256, 256, img.shape[2]), order=1, anti_aliasing=True, preserve_range=True)
    #print(img.shape)
    #print(img.min())

    #cropping anything larger than 224
    if height > new_size:
        img = img[(height//2)-(new_size//2):(height//2)+(new_size//2) ,: ,:]

    if width > new_size:
        img = img[:, (width//2)-(new_size//2):(width//2)+(new_size//2) ,:]

    #padding with minimum value if smaller than 224
    if height < new_size:

        pad_top = (new_size//2)-(height//2)
        pad_bottom = (new_size)-(height) - pad_top
        img = np.pad(img, ( (pad_top,pad_bottom), (0,0), (0,0) ), mode='constant', constant_values=padding_value)

    if width < new_size:

        pad_left = (new_size//2)-(width//2)
        pad_right = (new_size)-(width) - pad_left
        img = np.pad(img, ( (0,0), (pad_left,pad_right), (0,0) ), mode='constant', constant_values=padding_value)

    return img

#normalizing
#input an image and output normalized image from 0 to 1
def normalize(img):
    img = (img - img.min()) / (img.max() - img.min())
    return img

#generating the slices and saving them as numpy arrays
#input the file list, the path of original images and the save path
def dataset_generator(img_files, label_files, img_path, label_path, img_save_path, label_save_path):
    for nscan, (img_file, label_file) in enumerate(zip(img_files, label_files)):

        img = nib.load(os.path.join(img_path, img_file))
        label = nib.load(os.path.join(label_path, label_file))

        np_img = np.asanyarray(img.dataobj, dtype=np.float32)
        np_label = np.asanyarray(label.dataobj, dtype=np.float32)

        
        np_img = apply_windowing(np_img, W=1800, L=400)
        np_img = normalize(np_img)
        np_label = np_label>0
        np_label = np_label.astype(np.float32)

        np_img = reshape(np_img)
        np_label = reshape(np_label)

        #print(np_img.shape)

        if randint(0, 1) == 0:
            np_img = np.flip(np_img, axis=0)
            np_label = np.flip(np_label, axis=0)
        if randint(0, 1) == 0:
            np_img = np.flip(np_img, axis=1)
            np_label = np.flip(np_label, axis=1)

        for nslice in range(np_img.shape[2]):
            np.save(f"{img_save_path}image_{nscan+1}_{nslice+1}.npy", np_img[:,:,nslice])
            break
        for nslice in range(np_label.shape[2]):
            np.save(f"{label_save_path}image_{nscan+1}_{nslice+1}.npy", np_label[:,:,nslice])
            break
        break


#generating the patches and saving them as numpy arrays
#input the file list, the path of original images and the save path
def dataset_generator_patches(img_files, label_files, img_path, label_path, img_save_path, label_save_path, test_istrue=False):
    lesion_number = 0
    for nscan, (img_file, label_file) in enumerate(zip(img_files, label_files)):
        
        #out_of_bounds = 0
        
        img = nib.load(os.path.join(img_path, img_file))
        label = nib.load(os.path.join(label_path, label_file))

        np_img = np.asanyarray(img.dataobj, dtype=np.float32)
        np_label = np.asanyarray(label.dataobj, dtype=np.float32)

        np_img = apply_windowing(np_img, W=1800, L=400)
        np_img = normalize(np_img)
        np_label = np_label>0
        np_label = np_label.astype(np.float32)

        slice_list = []
        #going over each axial slice and getting ones with lesions
        for slice in range(np_img.shape[2]):

            img_slice = np_img[:, :, slice]
            label_slice = np_label[:, :, slice]

            #getting the labels
            labels = measure.label(label_slice, connectivity=1)

            #adding slice to list if there is lesion in slice
            if labels.max() != 0:

                slice_list.append(slice)

        for slice in slice_list:

            img_slice = np_img[:,:,slice]
            label_slice = np_label[:,:,slice]
            sk_labels = measure.label(label_slice)
            props = measure.regionprops(sk_labels)

            for prop in props:
                #print(prop.centroid[0])
                #check if the region plus the added number are within image bounds
                #if prop.centroid[0] - 112 > 0 and prop.centroid[0] + 112 < img_slice.shape[0] and prop.centroid[1] - 112 > 0 and prop.centroid[1] + 112 < img_slice.shape[1]:
                    #pass
                #else:
                    #out_of_bounds += 1

                half_patch_size = 224
                image_size = 768
                #get a random deviation from the centroid
                y_drift = random.randint(-64, 64)
                x_drift = random.randint(-64, 64)

                centroid_y = prop.centroid[0]
                centroid_x = prop.centroid[1]

                pad_up = 0
                pad_down = 0
                pad_left = 0
                pad_right = 0

                #padding if out of bounds

                if centroid_y + half_patch_size + y_drift > img_slice.shape[0]:
                    pad_down = int(abs((half_patch_size+y_drift+centroid_y)-img_slice.shape[0])) + 1
                    #print(pad_down)

                if (centroid_y - half_patch_size) + y_drift < 0:
                    pad_up = int(abs((centroid_y-half_patch_size)+y_drift)) + 1
                    centroid_y += pad_up
                    #print(pad_up)

                if centroid_x + half_patch_size + x_drift > img_slice.shape[1]:
                    pad_right = int(abs((half_patch_size+x_drift+centroid_x)-img_slice.shape[1])) + 1
                    #print(pad_right)

                if (centroid_x - half_patch_size) + x_drift < 0:
                    pad_left = int(abs((centroid_x-half_patch_size)+x_drift)) + 1
                    centroid_x += pad_left
                    #print(pad_left)

                

                img_slice = np.pad(img_slice, ((pad_up, pad_down), (pad_left, pad_right)), mode='constant')
                label_slice = np.pad(label_slice, ((pad_up, pad_down), (pad_left, pad_right)), mode='constant')

                #crop the image to a patch of size 224 with the centroid in the middle and a random drift
                img_patch = img_slice[int(centroid_y)-half_patch_size+y_drift : int(centroid_y)+half_patch_size+y_drift , int(centroid_x)-half_patch_size+x_drift : int(centroid_x)+half_patch_size+x_drift]
                label_patch = label_slice[int(centroid_y)-half_patch_size+y_drift : int(centroid_y)+half_patch_size+y_drift , int(centroid_x)-half_patch_size+x_drift : int(centroid_x)+half_patch_size+x_drift]
                
                #flipping but not for test set
                if test_istrue == False:
                    if randint(0, 1) == 0:
                        img_patch = np.flip(img_patch, axis=0)
                        label_patch = np.flip(label_patch, axis=0)
                    if randint(0, 1) == 0:
                        img_patch = np.flip(img_patch, axis=1)
                        label_patch = np.flip(label_patch, axis=1)

                if test_istrue == True:
                    img_patch = np.flipud(img_patch.T)
                    label_patch = np.flipud(label_patch.T)

                lesion_number += 1
                np.save(f"{img_save_path}image_{nscan+1}_{slice+1}_{lesion_number}.npy", img_patch)
                np.save(f"{label_save_path}image_{nscan+1}_{slice+1}_{lesion_number}.npy", label_patch)
            
                #break
            #break
        #break
                
        print(f"Scan number {nscan+1}")
    print(f"Number of lesions found: {lesion_number}")

In [ ]:
# #making datasets 448 patches
# dataset_generator_patches(img_train, label_train, img_path, label_path, "/wecare/home/daniel/Thesis/Dataset/448_patches/images/train/", "/wecare/home/daniel/Thesis/Dataset/448_patches/labels/train/")
# dataset_generator_patches(img_val, label_val, img_path, label_path, "/wecare/home/daniel/Thesis/Dataset/448_patches/images/validation/", "/wecare/home/daniel/Thesis/Dataset/448_patches/labels/validation/")
# dataset_generator_patches(img_test, label_test, img_path, label_path, "/wecare/home/daniel/Thesis/Dataset/448_patches/images/test/", "/wecare/home/daniel/Thesis/Dataset/448_patches/labels/test/", test_istrue=True)


Scan number 1
Scan number 2
Scan number 3
Scan number 4
Scan number 5
Scan number 6
Scan number 7
Scan number 8
Scan number 9
Scan number 10
Scan number 11
Scan number 12
Scan number 13
Scan number 14
Scan number 15
Scan number 16
Scan number 17
Scan number 18
Scan number 19
Scan number 20
Scan number 21
Scan number 22
Scan number 23
Scan number 24
Scan number 25
Scan number 26
Scan number 27
Scan number 28
Scan number 29
Scan number 30
Scan number 31
Scan number 32
Scan number 33
Scan number 34
Scan number 35
Scan number 36
Scan number 37
Scan number 38
Scan number 39
Scan number 40
Scan number 41
Scan number 42
Scan number 43
Scan number 44
Scan number 45
Scan number 46
Scan number 47
Scan number 48
Scan number 49
Scan number 50
Scan number 51
Scan number 52
Scan number 53
Scan number 54
Scan number 55
Scan number 56
Scan number 57
Scan number 58
Number of lesions found: 2174
Scan number 1
Scan number 2
Scan number 3
Scan number 4
Scan number 5
Scan number 6
Scan number 7
Scan number

In [ ]:
#making datasets 224 patches
# dataset_generator_patches(img_train, label_train, img_path, label_path, "/wecare/home/daniel/Thesis/Dataset/224_patches/images/train/", "/wecare/home/daniel/Thesis/Dataset/224_patches/labels/train/")
# dataset_generator_patches(img_val, label_val, img_path, label_path, "/wecare/home/daniel/Thesis/Dataset/224_patches/images/validation/", "/wecare/home/daniel/Thesis/Dataset/224_patches/labels/validation/")
# dataset_generator_patches(img_test, label_test, img_path, label_path, "/wecare/home/daniel/Thesis/Dataset/224_patches/images/test/", "/wecare/home/daniel/Thesis/Dataset/224_patches/labels/test/", test_istrue=True)


Scan number 1
Scan number 2
Scan number 3
Scan number 4
Scan number 5
Scan number 6
Scan number 7
Scan number 8
Scan number 9
Scan number 10
Scan number 11
Scan number 12
Scan number 13
Scan number 14
Scan number 15
Scan number 16
Scan number 17
Scan number 18
Scan number 19
Scan number 20
Scan number 21
Scan number 22
Scan number 23
Scan number 24
Scan number 25
Scan number 26
Scan number 27
Scan number 28
Scan number 29
Scan number 30
Scan number 31
Scan number 32
Scan number 33
Scan number 34
Scan number 35
Scan number 36
Scan number 37
Scan number 38
Scan number 39
Scan number 40
Scan number 41
Scan number 42
Scan number 43
Scan number 44
Scan number 45
Scan number 46
Scan number 47
Scan number 48
Scan number 49
Scan number 50
Scan number 51
Scan number 52
Scan number 53
Scan number 54
Scan number 55
Scan number 56
Scan number 57
Scan number 58
Number of lesions found: 2174
Scan number 1
Scan number 2
Scan number 3
Scan number 4
Scan number 5
Scan number 6
Scan number 7
Scan number